# 19 | LangChain Middleware：模型和工具失败时怎么兜底

Agent 跑在线业务时，失败不是小概率事件。

模型服务可能超时，订单 API 可能抖一下，第三方接口可能突然慢半拍。不能一次失败就让 Agent 对用户说：“我裂开了。”

这篇只讲可靠性兜底：能重试就重试，主模型不行就切备用。

本篇会做三件事：

- 模拟订单接口前两次超时，用 `ToolRetryMiddleware` 自动重试到成功
- 用本地 Ollama `qwen2.5:14b` 演示 `ModelRetryMiddleware` 如何重试同一个模型
- 用本地 Ollama 做主模型、云端 `deepseek-v4-flash` 做备用模型，演示 `ModelFallbackMiddleware` 如何切换

对应会用到三个官方 Middleware：

- `ToolRetryMiddleware`：工具失败自动重试
- `ModelRetryMiddleware`：模型失败自动重试
- `ModelFallbackMiddleware`：主模型失败时切备用模型

## 一、先准备模型和一个会超时的订单工具

场景还是客服查订单。

真实系统里，查订单通常要访问订单服务或数据库。偶发超时很正常，尤其是高峰期。我们先写一个“前两次超时，第三次成功”的工具，用来观察重试效果。

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv(dotenv_path=".env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)


In [ ]:
ORDER_API_ATTEMPTS = {"O-1001": 0}


@tool
def get_order_detail(order_id: str) -> dict:
    """根据订单号查询订单详情。"""
    # 模拟接口偶发超时：前两次失败，第三次成功。
    ORDER_API_ATTEMPTS[order_id] = ORDER_API_ATTEMPTS.get(order_id, 0) + 1
    attempt = ORDER_API_ATTEMPTS[order_id]

    print(f"[订单接口] 第 {attempt} 次查询订单 {order_id}")

    if attempt < 3:
        print(f"[订单接口] 查询失败：接口超时")
        raise TimeoutError(f"订单接口超时，第 {attempt} 次调用失败")

    print(f"[订单接口] 查询成功")
    return {
        "order_id": order_id,
        "item": "蓝牙耳机",
        "status": "未发货",
        "refund_policy": "未发货订单可以直接退款，1 到 3 个工作日原路退回。",
    }


## 二、工具失败：让它自动重试

没有重试时，工具第一次超时，Agent 很可能就拿不到订单信息。

`ToolRetryMiddleware` 的作用是：工具失败后，按规则再试几次。它适合处理**临时性失败**，比如超时、网络抖动、接口短暂不可用。

为了让效果更明显，工具里会打印每次调用记录。正常情况下，你会看到前两次失败，第三次成功。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware

ORDER_API_ATTEMPTS = {"O-1001": 0}

tool_retry_agent = create_agent(
    model=model,
    tools=[get_order_detail],
    middleware=[
        ToolRetryMiddleware(
            # 只给 get_order_detail 这个工具加重试，不影响其他工具。
            tools=["get_order_detail"],
            # 失败后最多再试 2 次：首次调用 + 2 次重试 = 最多调用 3 次。
            max_retries=2,
            # 只有 TimeoutError 才重试，参数错、权限错这类问题不要盲目重试。
            retry_on=(TimeoutError,),
            # 重试后仍失败时，把失败信息交给 Agent 继续处理。
            on_failure="continue",
            # 第一次重试前等 0.2 秒；如果继续失败，后面会等得更久一点。
            initial_delay=0.2,
            # 单次重试最多等待 1 秒，避免越等越久。
            max_delay=1.0,
        ),
    ],
    system_prompt=(
        "你是一个电商客服 Agent。"
        "回答订单退款问题前，先调用 get_order_detail 查询订单详情。"
        "回答要简洁说明订单状态和退款规则。"
    ),
)


In [ ]:
response = tool_retry_agent.invoke({
    "messages": [
        {"role": "user", "content": "订单 O-1001 能退款吗？多久到账？"}
    ]
})

print(response["messages"][-1].content)


这段代码的关键不是“第三次成功”，而是：**重试逻辑没有写在业务工具里，而是交给 Middleware 统一处理。**

这样工具函数只负责查订单，Middleware 负责失败兜底。职责分开，后面才不会到处散落 `try / except / sleep / retry`。

## 三、模型失败：让模型调用自动重试

工具会超时，模型服务也会。

这里用本地 Ollama 的 `qwen2.5:14b` 做主角。正常情况下它会直接回答；如果 Ollama 服务刚启动、短暂不可用、连接抖动，`ModelRetryMiddleware` 会对**同一个模型调用**做自动重试。

注意：模型重试不是切换模型。它做的是“同一个模型，再试几次”。切备用模型放到下一节。

In [ ]:
from langchain.agents.middleware import ModelRetryMiddleware, wrap_model_call

# 本地模型：需要本机 Ollama 服务可用，并且已拉取 qwen2.5:14b。
local_qwen_model = init_chat_model("ollama:qwen2.5:14b")


@wrap_model_call
def log_model_call(request, handler):
    """打印每一次模型调用尝试。"""
    print("[模型调用] 准备调用本地 qwen2.5:14b")
    try:
        result = handler(request)
        print("[模型调用] 调用成功")
        return result
    except Exception as e:
        print(f"[模型调用] 调用失败：{type(e).__name__}")
        raise


model_retry_agent = create_agent(
    model=local_qwen_model,
    tools=[],
    middleware=[
        ModelRetryMiddleware(
            # 模型调用失败后最多再试 10 次。
            # 如果想手动测试“关闭 Ollama 后再启动”的恢复过程，可以临时调大。
            max_retries=10,
            # 这里只重试连接失败、超时这类临时异常。
            retry_on=(Exception,),
            # 重试后仍失败时，把失败信息交给 Agent 继续处理。
            on_failure="continue",
            # 第一次重试前等 0.2 秒；如果继续失败，后面会等得更久一点。
            initial_delay=0.2,
            max_delay=1.0,
        ),
        # 放在 ModelRetryMiddleware 后面，可以打印每一次重试尝试。
        log_model_call,
    ],
    system_prompt="你是一个简洁的客服助手。",
)


In [ ]:
response = model_retry_agent.invoke({
    "messages": [
        {"role": "user", "content": "请用一句话说明：模型调用为什么需要重试？"}
    ]
})

final_message = response["messages"][-1]
print(final_message.content)
print("model_provider:", final_message.response_metadata.get("model_provider"))
print("model_name:", final_message.response_metadata.get("model_name"))


为了测试模型重试效果，可以临时关闭本地 Ollama 服务，或者把模型名改成一个本地不存在的模型。

模型服务有问题时，会看到多次调用失败记录；服务恢复后，后续某次重试会调用成功。

你实测时把 `max_retries` 临时调大到 20，这个方法很好，能模拟“服务恢复前一直失败，服务恢复后自动成功”的过程。不过文章里的默认值建议保留小一点，避免读者一运行就等太久。

## 四、主模型失败：切备用模型

重试解决的是“再试一次可能好”的问题。

Fallback 解决的是：**主模型已经不行了，换一个模型继续。**

这里继续使用同一组模型：

- 主模型：本地 Ollama 的 `qwen2.5:14b`
- 备用模型：云端 `deepseek-v4-flash`

正常情况下，Agent 优先使用本地模型；如果 Ollama 服务关闭、模型没拉取、连接失败，就切到云端备用模型。

In [ ]:
from langchain.agents.middleware import ModelFallbackMiddleware

# 备用模型：云端 DeepSeek。这里复用文章开头创建的 model。
cloud_deepseek_model = model

fallback_agent = create_agent(
    # Agent 会先尝试调用本地 qwen。
    model=local_qwen_model,
    tools=[],
    middleware=[
        # 如果本地 qwen 调用失败，再切到云端 deepseek-v4-flash。
        ModelFallbackMiddleware(cloud_deepseek_model),
    ],
    system_prompt="你是一个简洁的客服助手。",
)


In [ ]:
response = fallback_agent.invoke({
    "messages": [
        {"role": "user", "content": "请用一句话说明：为什么系统要有备用模型？"}
    ]
})

final_message = response["messages"][-1]
print(final_message.content)

# 打印实际返回的模型信息，方便确认这次回答来自本地模型还是备用模型。
print("model_provider:", final_message.response_metadata.get("model_provider"))
print("model_name:", final_message.response_metadata.get("model_name"))


如果 Ollama 正常运行，通常会看到类似：

```text
model_provider: ollama
model_name: qwen2.5:14b
```

如果你临时关闭 Ollama 服务，或者本地没有 `qwen2.5:14b`，主模型调用会失败，`ModelFallbackMiddleware` 会尝试切到备用模型。此时输出更可能类似：

```text
model_provider: openai
model_name: deepseek-v4-flash
```

两节放在一起看，区别就很清楚：

```text
ModelRetryMiddleware：同一个模型失败后，再试几次
ModelFallbackMiddleware：主模型失败后，换备用模型
```

Fallback 不是为了“多模型炫技”，而是为了线上系统别因为一个模型服务抖动就全线趴下。

## 五、哪些错误适合重试

不是所有错误都应该重试。

适合重试的，一般是临时性问题：

- 网络超时
- 连接失败
- 临时限流
- 服务短暂不可用
- 第三方接口偶发 5xx

不适合重试的，一般是确定性问题：

- 参数格式错误
- 订单号不存在
- 权限不足
- API Key 配错
- 工具逻辑本身写错

简单判断：**如果再试一次有可能成功，就重试；如果再试十次也只会错得更坚定，就别重试。**

## 六、怎么选：retry 还是 fallback

可以按这个思路选：

| 场景 | 更适合 |
|---|---|
| 工具偶发超时 | `ToolRetryMiddleware` |
| 本地 Ollama 偶发连接失败 | `ModelRetryMiddleware` |
| 本地 Ollama 持续不可用，需要切云端 | `ModelFallbackMiddleware` |
| 参数错、权限错、业务规则不允许 | 不要重试，直接暴露或转人工 |

如果只记一句话：**Retry 解决临时抖动，Fallback 解决主服务不可用。别把确定性错误拿去重试，那只是把错误重复播放。**

下一篇继续看 Human-in-the-loop：关键工具调用前，先让人点头。

参考：

- LangChain Prebuilt Middleware：https://docs.langchain.com/oss/python/langchain/middleware/built-in
- LangChain Middleware Overview：https://docs.langchain.com/oss/python/langchain/middleware/overview